In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita visualizacao grafica inline no Jupyter notebook
%matplotlib inline

# Salvar e recarregar dados preparados

**Dificuldade 1-2** | **Tempo de execucao: 5s** | **Computacao: CPU**

Pre-processar EEG e computacionalmente intensivo. Filtrar, reamostrar e janelar o sinal pode levar muito tempo; repetir isso a cada reinicio de kernel consome horas desnecessarias. O objetivo deste tutorial e demonstrar como persistir dados preparados em disco para acelerar sessoes futuras de analise.

Armazenamento em cache nao e apenas velocidade: um arquivo em cache sobrevive ao codigo que o gravou. E essencial saber qual versao da biblioteca, qual semente e qual commit geraram o array em disco. Este tutorial aborda: formato FIF para sinais janelados (Larson et al. 2024 / MNE), Apache Parquet para tabelas de caracteristicas e Zarr opcional para leitura em blocos de acesso aleatorio. A licao encerra com um painel comparando tempos de gravacao, leitura e uso em disco.

# Valide seu resultado
--------------------
- **Artefatos em cache:** Apos a execucao, arquivos ``.fif``, ``.parquet`` ou ``.zarr`` devem existir no diretorio.
- **Integridade dos dados:** Janelas recarregadas devem ser identicas as originais em formato e valores numéricos (validado com ``np.allclose``).
- **Velocidade de leitura:** A leitura a partir do cache aquecido deve ser ordens de grandeza mais rapida que o pre-processamento inicial.

.. sphinx_gallery_thumbnail_path = '_static/thumbs/plot_13_save_and_reuse_prepared_data.png'
Palavras-chave: offline, cache, E/S (I/O)


## Objetivos de aprendizagem
- Salvar um :class:`~braindecode.datasets.BaseConcatDataset` janelado com :meth:`~braindecode.datasets.BaseConcatDataset.save` e recarrega-lo com :func:`braindecode.datautil.load_concat_dataset`.
- Gravar uma tabela de caracteristicas por janela com :meth:`pandas.DataFrame.to_parquet` e recarrega-la com :func:`pandas.read_parquet`, preservando tipos de dados.
- Verificar que as janelas salvas sobrevivem ao ciclo completo com precisao numerica idêntica (dentro do limite float32).
- Registrar a proveniencia completa (versoes de pacotes, semente, commit git) para reprodutibilidade :cite:`wilkinson2016fair`.
- Interpretar o custo de cada formato a partir de uma figura consolidada.



Configuracao inicial. Usamos sinal sintetico para execucao rapida e totalmente offline.



In [ ]:
# Importa utilitarios de sistema, temporizadores e caminhos
from __future__ import annotations

import os
import shutil
import subprocess
import tempfile
import time
from pathlib import Path

# Importa bibliotecas graficas, MNE e computacao matricial
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

# Importa datasets e rotinas de carregamento e janelamento do Braindecode
import braindecode
import eegdash
from braindecode.datasets import BaseConcatDataset, RawDataset
from braindecode.datautil import load_concat_dataset
from braindecode.preprocessing import create_fixed_length_windows
from eegdash.viz import use_eegdash_style

# Aplica tema visual padronizado e fixa semente pseudoaleatoria
use_eegdash_style()
SEED = 42
np.random.seed(SEED)
mne.set_log_level("ERROR")
print(
    f"eegdash {eegdash.__version__} | braindecode {braindecode.__version__} | "
    f"numpy {np.__version__}"
)

## Etapa 1: Construir um pequeno conjunto de dados janelado
Sintetizamos um sinal de EEG em repouso com 2 canais e 4 segundos a 100 Hz, dividindo-o em duas janelas de 2 segundos sem sobreposicao.



In [ ]:
# Define frequencia de amostragem e tamanho da janela em segundos
SFREQ, WIN_S = 100, 2
# Gera matriz de sinal sintetico de 2 canais por 400 amostras em microvolts (float32)
signal = np.random.randn(2, 4 * SFREQ).astype("float32") * 1e-6
# Cria estrutura Info do MNE para os eletrodos Cz e Pz
info = mne.create_info(["Cz", "Pz"], sfreq=SFREQ, ch_types="eeg")
# Empacota o sinal no RawDataset da Braindecode
recording = RawDataset(
    mne.io.RawArray(signal, info),
    description={"subject": "S01", "task": "rest"},
)
# Cria janelas de comprimento fixo de 200 amostras com 0% de sobreposicao
windows = create_fixed_length_windows(
    BaseConcatDataset([recording]),
    start_offset_samples=0,
    stop_offset_samples=None,
    window_size_samples=WIN_S * SFREQ,
    window_stride_samples=WIN_S * SFREQ,
    drop_last_window=True,
    preload=True,
)
n_windows = len(windows)
sample_shape = windows[0][0].shape
n_channels, window_samples = int(sample_shape[0]), int(sample_shape[1])
# Exibe informacoes sobre as janelas geradas
pd.Series(
    {
        "n_windows": n_windows,
        "windows[0][0].shape": str(tuple(sample_shape)),
        "X.dtype": str(np.asarray(windows[0][0]).dtype),
        "child class": type(windows.datasets[0]).__name__,
    },
    name="value",
).to_frame()

## Etapa 2: Salvar as janelas no formato FIF
O metodo ``save`` grava os arquivos ``.fif`` e metadados auxiliares em formato JSON em um diretorio temporario.



In [ ]:
# Cria diretorio temporario seguro para testes de persistencia
cache_root = Path(tempfile.mkdtemp(prefix="eegdash_save_"))
windows_path = cache_root / "windows"

# Mede o tempo de gravacao do conjunto janelado em formato FIF
t0 = time.perf_counter()
windows.save(str(windows_path), overwrite=True)
fif_write_s = time.perf_counter() - t0


# Funcao para calcular o tamanho total em bytes do diretorio
def _dir_size_bytes(path: Path) -> int:
    """Calcula o tamanho total em bytes de um diretorio recursivamente."""
    total = 0
    for root, _, files in os.walk(path):
        for name in files:
            total += (Path(root) / name).stat().st_size
    return total


# Converte tamanho em megabytes e lista arquivos gravados
fif_size_mb = _dir_size_bytes(windows_path) / 1e6
saved_files = sorted(
    p.relative_to(cache_root).as_posix() for p in windows_path.rglob("*")
)
print(f"saved: {windows_path}")
print(f"artifact tree (first 6): {saved_files[:6]}")
print(f"FIF write_s={fif_write_s:.4f} s, size_mb={fif_size_mb:.4f}")

## Etapa 3: Recarregar as janelas
Recarregamos o objeto salvo com ``load_concat_dataset`` e validamos a correspondencia exata dos valores numericos.



In [ ]:
# Mede o tempo de recarregamento dos dados persistidos
t0 = time.perf_counter()
reloaded_fif = load_concat_dataset(windows_path, preload=True)
fif_read_s = time.perf_counter() - t0
print(
    f"reload OK: type={type(reloaded_fif).__name__}, n={len(reloaded_fif)}, "
    f"read_s={fif_read_s:.4f}"
)

In [ ]:
# Compara as amostras originais e recarregadas ponto a ponto
x_orig = np.asarray(windows[0][0]).copy()
x_re = np.asarray(reloaded_fif[0][0]).copy()
residual = x_re - x_orig
# Validacoes de integridade estrutural e numerica
assert len(reloaded_fif) == n_windows, "reloaded window count differs"
assert x_re.shape == sample_shape, "reloaded window shape differs"
assert np.allclose(x_re, x_orig, atol=1e-7), "samples drifted beyond float32 tol"
assert list(reloaded_fif.description.columns) == list(windows.description.columns)
print(
    f"shapes match: original={sample_shape}, reloaded={x_re.shape}; "
    f"max|residual|={float(np.max(np.abs(residual))):.2e}"
)

## Etapa 4: Cache Zarr opcional
O formato Zarr permite leitura aleatoria em blocos para bases de dados de grandes dimensoes.



In [ ]:
# Verifica disponibilidade do modulo Zarr na instalacao atual
try:
    BaseConcatDataset._convert_to_zarr_inline  # noqa: B018 - teste de recurso
    has_zarr = True
except (AttributeError, ImportError):
    has_zarr = False

zarr_record = None
if has_zarr:
    zarr_path = cache_root / "windows.zarr"
    try:
        t0 = time.perf_counter()
        windows._convert_to_zarr_inline(
            zarr_path,
            compression="blosc",
            compression_level=5,
            chunk_size=5_000_000,
        )
        zarr_write_s = time.perf_counter() - t0

        t0 = time.perf_counter()
        reloaded_zarr = type(windows)._load_from_zarr_inline(zarr_path, preload=True)
        zarr_read_s = time.perf_counter() - t0
        zarr_size_mb = _dir_size_bytes(zarr_path) / 1e6
        zarr_record = {
            "name": "windows.zarr (Zarr)",
            "write_s": zarr_write_s,
            "read_s": zarr_read_s,
            "size_mb": zarr_size_mb,
        }
        print(
            f"Zarr write_s={zarr_write_s:.4f}, read_s={zarr_read_s:.4f}, "
            f"size_mb={zarr_size_mb:.4f}"
        )
    except (ImportError, RuntimeError) as exc:
        has_zarr = False
        print(f"Zarr extra unavailable, skipping: {type(exc).__name__}: {exc}")
else:
    print("Zarr extra not installed (pip install braindecode[hub]); skipping.")

## Etapa 5: Salvar e recarregar tabela de caracteristicas em Parquet
O formato colunar Parquet armazena caracteristicas tabulares de forma compacta e com tipos de dados preservados.



In [ ]:
# Monta DataFrame de caracteristicas simples extraidas por janela
features = pd.DataFrame(
    [
        {
            "Cz_mean": float(windows[i][0][0].mean()),
            "Pz_mean": float(windows[i][0][1].mean()),
            "window_idx": i,
        }
        for i in range(n_windows)
    ]
)
features_path = cache_root / "features.parquet"
# Grava tabela em formato Parquet medindo o tempo
t0 = time.perf_counter()
features.to_parquet(features_path, index=False)
parquet_write_s = time.perf_counter() - t0

# Recarrega tabela a partir do Parquet
t0 = time.perf_counter()
features_back = pd.read_parquet(features_path)
parquet_read_s = time.perf_counter() - t0

# Valida igualdade exata entre as tabelas gravada e recarregada
pd.testing.assert_frame_equal(features_back, features)
parquet_size_mb = features_path.stat().st_size / 1e6
print(f"feature table dtype:\n{features.dtypes.to_string()}")
print(
    f"Parquet write_s={parquet_write_s:.4f}, read_s={parquet_read_s:.4f}, "
    f"size_mb={parquet_size_mb:.4f}"
)
features.head()

## Etapa 6: Consolidar comparativo de formatos



In [ ]:
# Consolida tempos e tamanhos medidos para cada formato
format_records = [
    {
        "name": "windows/ (FIF)",
        "write_s": fif_write_s,
        "read_s": fif_read_s,
        "size_mb": fif_size_mb,
    },
]
if zarr_record is not None:
    format_records.append(zarr_record)
format_records.append(
    {
        "name": "features.parquet",
        "write_s": parquet_write_s,
        "read_s": parquet_read_s,
        "size_mb": parquet_size_mb,
    }
)
records_df = pd.DataFrame(format_records)
records_df["write_ms"] = (records_df["write_s"] * 1000).round(2)
records_df["read_ms"] = (records_df["read_s"] * 1000).round(2)
records_df[["name", "write_ms", "read_ms", "size_mb"]]

## Etapa 7: Selo de proveniencia



In [ ]:
# Funcao para capturar o commit do repositorio git
def _git_short_sha() -> str:
    """Retorna o hash abreviado do commit atual do Git."""
    try:
        result = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"],
            capture_output=True,
            text=True,
            timeout=2.0,
            check=False,
        )
        sha = (result.stdout or "").strip()
        return sha or "git: not available"
    except (OSError, subprocess.SubprocessError):
        return "git: not available"


# Registra versoes e parametros de ambiente
provenance = {
    "eegdash": eegdash.__version__,
    "braindecode": braindecode.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "mne": mne.__version__,
    "seed": str(SEED),
    "git": _git_short_sha(),
}
pd.Series(provenance, name="value").to_frame()

## Painel diagnostico e limpeza



In [ ]:
# Importa e exibe painel visual dos artefatos
from _ledger_figure import draw_ledger_figure

fig = draw_ledger_figure(
    format_records=format_records,
    residual_array=residual,
    provenance_dict=provenance,
    n_windows=n_windows,
    n_channels=n_channels,
    window_samples=window_samples,
    plot_id="plot_13",
)
plt.show()

In [ ]:
# Remove o diretorio temporario de testes
shutil.rmtree(cache_root, ignore_errors=True)
print("cleanup OK")

## Conclusao
Persistimos sinais e caracteristicas em formatos de alta eficiencia (FIF, Zarr e Parquet), demonstrando integridade e documentacao de proveniencia.

